# MedMNIST-C API debug

This notebook verifies the cloned API and its data flow. The smoke dataset is resized from two official 28-pixel PathMNIST test images only to exercise the API; its outputs must never be reported as robustness results. Full generation requires official `pathmnist_224.npz` or the corresponding official 224 files.

In [1]:
from pathlib import Path
import hashlib, json, os, shutil
import numpy as np
from PIL import Image

ROOT = Path('/project/prj-sis01/xuxiaoyu/reliability_medmnistc_ab')
REPO = ROOT / 'sources' / 'medmnistc-api'
SMOKE_CLEAN = ROOT / 'data' / 'medmnist_smoke'
SMOKE_CORRUPTED = ROOT / 'data' / 'medmnistc_smoke'
SMOKE_RESULTS = ROOT / 'results' / 'raw_metrics' / 'api_smoke'
SMOKE_CLEAN.mkdir(parents=True, exist_ok=True); SMOKE_CORRUPTED.mkdir(parents=True, exist_ok=True); SMOKE_RESULTS.mkdir(parents=True, exist_ok=True)
print(REPO)

/project/prj-sis01/xuxiaoyu/reliability_medmnistc_ab/sources/medmnistc-api


In [2]:
import os, subprocess, sys
sys.dont_write_bytecode = True
assert (REPO / 'medmnistc' / 'dataset_manager.py').exists()
commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
status = subprocess.check_output(['git', '-C', str(REPO), 'status', '--short'], text=True)
code_dirty = [line for line in status.splitlines() if '__pycache__' not in line]
assert not code_dirty, code_dirty
print({'commit': commit, 'python': sys.version, 'source_code_clean': True, 'cache_status_before_import': status.splitlines()})

{'commit': '8acfd2710c6e0e8b2745be8b1fa1c17b94b8f8a7', 'python': '3.11.9 | packaged by conda-forge | (main, Apr 19 2024, 18:36:13) [GCC 12.3.0]', 'source_code_clean': True, 'cache_status_before_import': [' M medmnistc/corruptions/__pycache__/__init__.cpython-311.pyc', ' M medmnistc/corruptions/__pycache__/base.cpython-311.pyc', ' M medmnistc/corruptions/__pycache__/noise.cpython-311.pyc', ' M medmnistc/corruptions/__pycache__/registry.cpython-311.pyc']}


In [3]:
from medmnist import PathMNIST
source_root = Path('/project/prj-sis01/xuxiaoyu/HOP/datasets/PathMNIST')
source = np.load(source_root / 'pathmnist.npz')
imgs = source['test_images'][:2]
labels = source['test_labels'][:2]
resized = np.stack([np.asarray(Image.fromarray(x).resize((224, 224), Image.Resampling.BILINEAR)) for x in imgs])
np.savez_compressed(SMOKE_CLEAN / 'pathmnist_224.npz', train_images=resized, val_images=resized, test_images=resized, train_labels=labels, val_labels=labels, test_labels=labels)
print({'synthetic_smoke_shape': resized.shape, 'source_shape': imgs.shape})

{'synthetic_smoke_shape': (2, 224, 224, 3), 'source_shape': (2, 28, 28, 3)}


In [4]:
from medmnistc.dataset_manager import DatasetManager
from medmnistc.dataset import CorruptedMedMNIST
from medmnistc.corruptions.registry import CORRUPTIONS_DS
manager = DatasetManager(medmnist_path=str(SMOKE_CLEAN), output_path=str(SMOKE_CORRUPTED), random_seed=0)
manager.create_dataset('pathmnist')
corrs = list(CORRUPTIONS_DS['pathmnist'])
checks = {}
for corruption in corrs:
    ds = CorruptedMedMNIST('pathmnist', corruption, root=str(SMOKE_CORRUPTED), as_rgb=True)
    checks[corruption] = {'length': len(ds), 'sample_shape': list(ds[0][0].shape), 'labels': int(len(ds.labels))}
summary = {'kind': 'synthetic_api_smoke_only', 'corruptions': checks, 'source_commit': commit}
(SMOKE_RESULTS / 'medmnistc_api_smoke_summary.json').write_text(json.dumps(summary, indent=2))
print(summary)

=========== pathmnist ===========
Starting pixelate...


Severity 01:   0%|          | 0/2 [00:00<?, ?it/s]

Severity 01: 100%|██████████| 2/2 [00:00<00:00, 1201.81it/s]

Severity 02:   0%|          | 0/2 [00:00<?, ?it/s]

Severity 02: 100%|██████████| 2/2 [00:00<00:00, 1634.57it/s]

...

Severity 04:   0%|          | 0/2 [00:00<?, ?it/s]

Severity 04: 100%|██████████| 2/2 [00:00<00:00, 2239.95it/s]

Severity 05:   0%|          | 0/2 [00:00<?, ?it/s]

Severity 05: 100%|██████████| 2/2 [00:00<00:00, 2365.65it/s]

{'kind': 'synthetic_api_smoke_only', 'corruptions': {'pixelate': {'length': 10, 'sample_shape': [3, 224, 224], 'labels': 10}, 'jpeg_compression': {'length': 10, 'sample_shape': [3, 224, 224], 'labels': 10}, 'defocus_blur': {'length': 10, 'sample_shape': [3, 224, 224], 'labels': 10}, 'motion_blur': {'length': 10, 'sample_shape': [3, 224, 224], 'labels': 10}, 'brightness_up': {'length': 10, 'sample_shape': [3, 224, 224], 'labels': 10}, 'brightness_down': {'length': 10, 'sample_shape': [3, 224, 224], 'labels': 10}, 'contrast_up': {'length': 10, 'sample_shape': [3, 224, 224], 'labels': 10}, 'contrast_down': {'length': 10, 'sample_shape': [3, 224, 224], 'labels': 10}, 'saturate': {'length': 10, 'sample_shape': [3, 224, 224], 'labels': 10}, 'stain_deposit': {'length': 10, 'sample_shape': [3, 224, 224], 'labels': 10}, 'bubble': {'length': 10, 'sample_shape': [3, 224, 224], 'labels': 10}}, 'source_commit': '8acfd2710c6e0e8b2745be8b1fa1c17b94b8f8a7'}
